# GeoPulse - Retail Mix vs Mobility (Day 10)

**Question:** do hubs with a different shop mix (more food places, more fashion, denser retail) show a different taxi/bike pattern?

Inputs (created by earlier days):
- `reports/tables/osm_hub_summary.csv` - POIs around each hub (Day 4, OpenStreetMap)
- `reports/tables/hub_rhythm_stats_<month>.csv` - taxi/bike rhythm per hub (Days 8-9)

**Caution:** about ten hubs and one month. Everything below is exploratory, not proof.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from geopulse import retail_mobility as rm

# Find the project root (the folder that contains setup.py) from wherever the notebook runs
ROOT = Path.cwd()
while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
TABLES = ROOT / "reports" / "tables"
YM = "2024-01"          # change to analyse another month
pd.options.display.float_format = "{:,.2f}".format
print("Project root:", ROOT)

## 1. Load the data and build one feature table

In [ ]:
osm = pd.read_csv(TABLES / "osm_hub_summary.csv")
stats = pd.read_csv(TABLES / f"hub_rhythm_stats_{YM}.csv")
features = rm.hub_feature_table(osm, stats)

show = ["hub_name", "hub_type", "poi_total", "poi_per_km2", "dominant_category", "diversity",
        "taxi_total", "bike_total", "taxi_rhythm", "bike_rhythm"]
display(features[[c for c in show if c in features.columns]])

## 2. Retail mix around each hub
Each row is a hub; each cell is the % of that hub's POIs in the category. **Diversity** (0-1) is 1 when all categories are equally common.

In [ ]:
share_cols = [c for c in features.columns if c.startswith("share_")]
d = features.sort_values("poi_total").set_index("hub_name")[share_cols] * 100
d.columns = [c.replace("share_", "") for c in d.columns]

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(d.fillna(0).values, aspect="auto", cmap="Blues", vmin=0)
ax.set_xticks(range(d.shape[1]), d.columns, rotation=35, ha="right")
ax.set_yticks(range(d.shape[0]), d.index)
fig.colorbar(im, ax=ax, label="% of the hub's POIs")
ax.set_title("Retail mix around each hub")
plt.show()

## 3. Is retail density related to how many trips a hub gets?

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, pre in zip(axs, ("taxi", "bike")):
    col = f"{pre}_total"
    if col not in features.columns:
        ax.set_visible(False)
        continue
    sub = features.dropna(subset=[col])
    ax.scatter(sub["poi_per_km2"], sub[col])
    for r in sub.itertuples():
        ax.annotate(r.hub_name.split(" (")[0], (r.poi_per_km2, getattr(r, col)), xytext=(4, 3),
                    textcoords="offset points", fontsize=7)
    rho, n, p = rm.spearman(sub["poi_per_km2"], sub[col])
    ax.set_yscale("log")
    ax.set_xlabel("POIs per km2")
    ax.set_ylabel(f"{pre} trips in the month")
    ax.set_title(f"{pre}: rho={rho:.2f}, n={n}, p={p:.3f}")
plt.show()

## 4. Rank correlations (all pairs)
`rho` is Spearman's rank correlation. With ~10 hubs and dozens of pairs, a few strong values will appear by chance, so look for patterns that agree between taxi and bike.

In [ ]:
x_cols, y_cols = rm.default_columns(features)
corr = rm.correlation_table(features, x_cols, y_cols)
print(f"{corr['rho'].notna().sum()} correlations; about {corr['rho'].notna().sum() * 0.05:.0f} would reach p < 0.05 by chance alone")
display(corr.dropna(subset=["rho"]).head(12))

## 5. Mobility relative to retail supply
Trips per 100 POIs: a high value may point to a hub with a lot of movement for its amount of retail (an opportunity?) or to shops missing from OpenStreetMap. Check both.

In [ ]:
cols = [c for c in ("taxi_per_100_poi", "bike_per_100_poi") if c in features.columns]
display(features.set_index("hub_name")[cols].sort_values(cols[0], ascending=False))

## 6. My findings (write 3-5 sentences)
- Which hubs have the most / least retail per km2?
- Do denser or more diverse hubs get more trips, for both taxi and bike?
- Which hub has the highest trips per 100 POIs, and what could explain it?
- What would I check next with more months of data?